Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, random_split
from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

import pandas as pd
 

Creating the NN

In [2]:
class Net(nn.Module):
    def __init__(self, d_in):
        super(Net, self).__init__()

        self.fc1 = nn.Linear(d_in, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        x = F.gelu(self.fc3(x))
        x = self.fc4(x)
        return x


Load The Heston Model Parameters

In [3]:
df_parquet_1 = pd.read_parquet("exotic_mc_dataset_1.parquet")
df_heston_selected_parquet_1 = df_parquet_1[df_parquet_1["model"] == "HESTON"].loc[:, "option":"T"].join(
    df_parquet_1.loc[df_parquet_1["model"] == "HESTON", "v0":"rho"]
)

Y_parquet_1 = torch.tensor(df_heston_selected_parquet_1["price"].values, dtype=torch.float32).view(-1, 1)

opt_dummies_1 = pd.get_dummies(
    df_heston_selected_parquet_1["option"],
    prefix="opt"
).astype("int64")

num_cols = [
    "S0",
    "K",
    "r",
    "T",
    "v0",
    "kappa",
    "theta",
    "xi",
    "rho"
]

X_df_1 = pd.concat(
    [
        df_heston_selected_parquet_1[num_cols],
        opt_dummies_1
    ],
    axis=1
)

X_parquet_1 = torch.tensor(X_df_1.values, dtype=torch.float32)

X = torch.cat([X_parquet_1], dim=0)
Y = torch.cat([Y_parquet_1], dim=0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.numpy())
X = torch.tensor(X_scaled, dtype=torch.float32)

input_dim = X.shape[1]
torch.manual_seed(42)
net = Net(input_dim)

display(X.shape)
display(Y.shape)

torch.Size([3309, 15])

torch.Size([3309, 1])

In [4]:
X_df_1

,S0,K,r,T,v0,kappa,theta,xi,rho,opt_ASIAN_ARITH_CALL,opt_ASIAN_ARITH_PUT,opt_BARRIER_DOWN_AND_OUT_PUT,opt_BARRIER_UP_AND_OUT_CALL,opt_LOOKBACK_FLOAT_CALL,opt_LOOKBACK_FLOAT_PUT
3,102.485076,147.685871,0.027255,1.389348,0.494418,2.718432,0.364860,1.209757,-0.073835,0,1,0,0,0,0
6,82.847837,112.842249,0.000005,0.548326,0.478771,3.801205,0.424584,0.376359,-0.600336,0,0,1,0,0,0
7,63.109201,55.816582,0.019524,0.058221,0.366511,3.634015,0.398027,0.650938,-0.004666,0,0,1,0,0,0
10,147.768757,93.497025,0.005463,0.646335,0.343687,3.230841,0.441155,1.707285,-0.725269,0,0,0,1,0,0
11,103.091564,79.970279,0.040125,0.923979,0.139105,0.881485,0.163398,0.565200,-0.067560,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9992,96.479731,122.885592,0.078665,0.196752,0.014711,0.584423,0.287987,0.721452,-0.642265,0,0,0,1,0,0
9994,88.812921,62.188946,0.006207,1.817779,0.440693,2.061102,0.198395,0.961681,-0.131543,0,0,0,0,1,0
9995,64.857144,147.328215,0.004579,0.107087,0.330328,0.501510,0.280286,0.292201,-0.022449,0,0,0,0,1,0
9996,123.312902,130.558532,0.001917,0.316705,0.348388,3.492039,0.133702,1.415792,-0.444135,0,0,0,0,1,0


Setting the DataLoader                                    

In [5]:
dataset = TensorDataset(X, Y)

#Lets keep 70% and 30%
n_total = len(dataset)
n_train = int(0.7 * n_total)
n_test = n_total - n_train

train_dataset, test_dataset = random_split(dataset, [n_train, n_test])


train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False)
train_loader

Define a Loss function and optimizer

In [6]:
criterion = nn.MSELoss()
optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

Training the Network

In [7]:
for epoch in range(100):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 20 == 19:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 20:.3f}')
            running_loss = 0.0

print('Finished Training')

Finished Training


In [8]:
net.eval()

correct = 0
total = 0
tol = 0.05  # absolute tolerance

with torch.no_grad():
    for X, y in test_loader:
        preds = net(X).view_as(y)              # ensure shape matches (N,1)
        within = (preds - y).abs() <= tol      # boolean tensor
        correct += within.sum().item()
        total += y.numel()

acc = 100.0 * correct / total
print(f"Accuracy within ±{tol}: {acc:.2f}%")

Accuracy within ±0.05: 2.11%


In [9]:
net.eval()

correct = 0
total = 0
pct = 0.20  # 20%

eps = 1e-8  # avoid divide-by-zero
with torch.no_grad():
    for X, y in test_loader:
        preds = net(X).view_as(y)
        rel_err = (preds - y).abs() / (y.abs() + eps)
        within = rel_err <= pct
        correct += within.sum().item()
        total += y.numel()

acc = 100.0 * correct / total
print(f"Accuracy within ±{pct*100:.1f}%: {acc:.2f}%")

Accuracy within ±20.0%: 45.52%
